In [1]:
import pandas as pd

data = pd.read_csv("bank-additional-full.csv", sep=";")
print("Dataset Shape:", data.shape)
data.head()

Dataset Shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [2]:
# Convert target variable
data['y'] = data['y'].map({'yes': 1, 'no': 0})

print(data['y'].value_counts())

y
0    36548
1     4640
Name: count, dtype: int64


In [3]:
from sklearn.preprocessing import LabelEncoder

# Encode all categorical columns
label_encoders = {}

for col in data.select_dtypes(include='object').columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

print("Encoding complete.")
data.head()


Encoding complete.


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,3,1,0,0,0,0,1,6,1,...,1,999,0,1,1.1,93.994,-36.4,4.857,5191.0,0
1,57,7,1,3,1,0,0,1,6,1,...,1,999,0,1,1.1,93.994,-36.4,4.857,5191.0,0
2,37,7,1,3,0,2,0,1,6,1,...,1,999,0,1,1.1,93.994,-36.4,4.857,5191.0,0
3,40,0,1,1,0,0,0,1,6,1,...,1,999,0,1,1.1,93.994,-36.4,4.857,5191.0,0
4,56,7,1,3,0,0,2,1,6,1,...,1,999,0,1,1.1,93.994,-36.4,4.857,5191.0,0


In [4]:
# Separate features and target
X = data.drop('y', axis=1)
y = data['y']

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


Feature shape: (41188, 20)
Target shape: (41188,)


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (32950, 20)
Test shape: (8238, 20)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef

# Initialize model
log_model = LogisticRegression()

# Train
log_model.fit(X_train, y_train)

# Predict
y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("Logistic Regression Results:")
print("Accuracy:", acc)
print("AUC:", auc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)
print("MCC:", mcc)


Logistic Regression Results:
Accuracy: 0.9104151493080845
AUC: 0.9317171684627443
Precision: 0.6678023850085179
Recall: 0.4192513368983957
F1 Score: 0.5151116951379764
MCC: 0.4840310980920279


In [7]:
import joblib

joblib.dump(log_model, "logistic_regression.pkl")
print("Model saved successfully.")


Model saved successfully.


In [8]:
from sklearn.tree import DecisionTreeClassifier

# Initialize model
dt_model = DecisionTreeClassifier(random_state=42)

# Train
dt_model.fit(X_train, y_train)

# Predict
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

# Metrics
acc_dt = accuracy_score(y_test, y_pred_dt)
auc_dt = roc_auc_score(y_test, y_prob_dt)
prec_dt = precision_score(y_test, y_pred_dt)
rec_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
mcc_dt = matthews_corrcoef(y_test, y_pred_dt)

print("Decision Tree Results:")
print("Accuracy:", acc_dt)
print("AUC:", auc_dt)
print("Precision:", prec_dt)
print("Recall:", rec_dt)
print("F1 Score:", f1_dt)
print("MCC:", mcc_dt)

Decision Tree Results:
Accuracy: 0.8888079630978393
AUC: 0.722790648630956
Precision: 0.5102040816326531
Recall: 0.5080213903743316
F1 Score: 0.5091103965702036
MCC: 0.44641524283463424


In [9]:
joblib.dump(dt_model, "decision_tree.pkl")
print("Decision Tree model saved.")

Decision Tree model saved.


In [10]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize model
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train
knn_model.fit(X_train, y_train)

# Predict
y_pred_knn = knn_model.predict(X_test)
y_prob_knn = knn_model.predict_proba(X_test)[:, 1]

# Metrics
acc_knn = accuracy_score(y_test, y_pred_knn)
auc_knn = roc_auc_score(y_test, y_prob_knn)
prec_knn = precision_score(y_test, y_pred_knn)
rec_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)
mcc_knn = matthews_corrcoef(y_test, y_pred_knn)

print("KNN Results:")
print("Accuracy:", acc_knn)
print("AUC:", auc_knn)
print("Precision:", prec_knn)
print("Recall:", rec_knn)
print("F1 Score:", f1_knn)
print("MCC:", mcc_knn)

KNN Results:
Accuracy: 0.8997329448895363
AUC: 0.8569583520361201
Precision: 0.5858267716535434
Recall: 0.39786096256684494
F1 Score: 0.47388535031847134
MCC: 0.4303320699117937


In [11]:
joblib.dump(knn_model, "knn.pkl")
print("KNN model saved.")

KNN model saved.


In [12]:
from sklearn.naive_bayes import GaussianNB

# Initialize model
nb_model = GaussianNB()

# Train
nb_model.fit(X_train, y_train)

# Predict
y_pred_nb = nb_model.predict(X_test)
y_prob_nb = nb_model.predict_proba(X_test)[:, 1]

# Metrics
acc_nb = accuracy_score(y_test, y_pred_nb)
auc_nb = roc_auc_score(y_test, y_prob_nb)
prec_nb = precision_score(y_test, y_pred_nb)
rec_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)
mcc_nb = matthews_corrcoef(y_test, y_pred_nb)

print("Naive Bayes Results:")
print("Accuracy:", acc_nb)
print("AUC:", auc_nb)
print("Precision:", prec_nb)
print("Recall:", rec_nb)
print("F1 Score:", f1_nb)
print("MCC:", mcc_nb)

Naive Bayes Results:
Accuracy: 0.8505705268268997
AUC: 0.8497759692925256
Precision: 0.39764868603042874
Recall: 0.6149732620320856
F1 Score: 0.48299034019319614
MCC: 0.4133316899634482


In [13]:
joblib.dump(nb_model, "naive_bayes.pkl")
print("Naive Bayes model saved.")

Naive Bayes model saved.


In [14]:
from sklearn.ensemble import RandomForestClassifier

# Initialize model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# Metrics
acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
mcc_rf = matthews_corrcoef(y_test, y_pred_rf)

print("Random Forest Results:")
print("Accuracy:", acc_rf)
print("AUC:", auc_rf)
print("Precision:", prec_rf)
print("Recall:", rec_rf)
print("F1 Score:", f1_rf)
print("MCC:", mcc_rf)

Random Forest Results:
Accuracy: 0.9129643117261471
AUC: 0.9432622005021742
Precision: 0.6472972972972973
Recall: 0.5122994652406417
F1 Score: 0.5719402985074626
MCC: 0.5286717791938483


In [15]:
joblib.dump(rf_model, "random_forest.pkl")
print("Random Forest model saved.")

Random Forest model saved.


In [16]:
!pip install xgboost

In [17]:
from xgboost import XGBClassifier

# Initialize model
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train
xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Metrics
acc_xgb = accuracy_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)
mcc_xgb = matthews_corrcoef(y_test, y_pred_xgb)

print("XGBoost Results:")
print("Accuracy:", acc_xgb)
print("AUC:", auc_xgb)
print("Precision:", prec_xgb)
print("Recall:", rec_xgb)
print("F1 Score:", f1_xgb)
print("MCC:", mcc_xgb)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:54:46] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Results:
Accuracy: 0.9183054139354212
AUC: 0.9488183377866104
Precision: 0.6688144329896907
Recall: 0.5550802139037433
F1 Score: 0.6066627703097603
MCC: 0.5645582594766


In [18]:
joblib.dump(xgb_model, "xgboost.pkl")
print("XGBoost model saved.")

XGBoost model saved.


In [19]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "KNN",
        "Naive Bayes",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        acc, acc_dt, acc_knn, acc_nb, acc_rf, acc_xgb
    ],
    "AUC": [
        auc, auc_dt, auc_knn, auc_nb, auc_rf, auc_xgb
    ],
    "Precision": [
        prec, prec_dt, prec_knn, prec_nb, prec_rf, prec_xgb
    ],
    "Recall": [
        rec, rec_dt, rec_knn, rec_nb, rec_rf, rec_xgb
    ],
    "F1 Score": [
        f1, f1_dt, f1_knn, f1_nb, f1_rf, f1_xgb
    ],
    "MCC": [
        mcc, mcc_dt, mcc_knn, mcc_nb, mcc_rf, mcc_xgb
    ]
})

comparison

,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.910415,0.931717,0.667802,0.419251,0.515112,0.484031
1,Decision Tree,0.888808,0.722791,0.510204,0.508021,0.509110,0.446415
2,KNN,0.899733,0.856958,0.585827,0.397861,0.473885,0.430332
3,Naive Bayes,0.850571,0.849776,0.397649,0.614973,0.482990,0.413332
4,Random Forest,0.912964,0.943262,0.647297,0.512299,0.571940,0.528672
5,XGBoost,0.918305,0.948818,0.668814,0.555080,0.606663,0.564558


In [20]:
joblib.dump(scaler, "scaler.pkl")
print("Scaler saved.")

Scaler saved.


In [21]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

st.title("Bank Marketing Classification App")

# Upload CSV
uploaded_file = st.file_uploader("Upload Test CSV File", type=["csv"])

# Model selection
model_option = st.selectbox(
    "Select Model",
    ["Logistic Regression", "Decision Tree", "KNN", "Naive Bayes", "Random Forest", "XGBoost"]
)

if uploaded_file is not None:
    data = pd.read_csv(uploaded_file, sep=";")

    # Convert target
    data['y'] = data['y'].map({'yes': 1, 'no': 0})

    # Encode categorical
    from sklearn.preprocessing import LabelEncoder
    for col in data.select_dtypes(include='object').columns:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])

    X = data.drop('y', axis=1)
    y = data['y']

    # Load scaler
    scaler = joblib.load("scaler.pkl")
    X_scaled = scaler.transform(X)

    # Load model
    model_dict = {
        "Logistic Regression": "logistic_regression.pkl",
        "Decision Tree": "decision_tree.pkl",
        "KNN": "knn.pkl",
        "Naive Bayes": "naive_bayes.pkl",
        "Random Forest": "random_forest.pkl",
        "XGBoost": "xgboost.pkl"
    }

    model = joblib.load(model_dict[model_option])

    y_pred = model.predict(X_scaled)
    y_prob = model.predict_proba(X_scaled)[:, 1]

    # Metrics
    acc = accuracy_score(y, y_pred)
    auc = roc_auc_score(y, y_prob)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    mcc = matthews_corrcoef(y, y_pred)

    st.subheader("Evaluation Metrics")
    st.write("Accuracy:", acc)
    st.write("AUC:", auc)
    st.write("Precision:", prec)
    st.write("Recall:", rec)
    st.write("F1 Score:", f1)
    st.write("MCC:", mcc)

    # Confusion Matrix
    st.subheader("Confusion Matrix")
    cm = confusion_matrix(y, y_pred)
    fig, ax = plt.subplots()
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", ax=ax)
    st.pyplot(fig)

Writing app.py


In [22]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.3 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.0
    Uninstalling cachetools-7.0.0:
      Successfully uninstalled cachetools-7.0.0


In [23]:
#!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.15.67:8501

  Stopping...
^C


In [24]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
matplotlib
seaborn
xgboost
joblib

Writing requirements.txt
